# Mode Report

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [2]:
import gcsfs
import pandas as pd
import numpy as np

import process_ntd
import mode_report_chart
import altair as alt
alt.data_transformers.enable("vegafusion");

/home/jovyan/ntd-snapshot/.venv/lib/python3.11/site-packages/google/api_core/_python_version_support.py:255: FutureWarning: You are using a Python version (3.11.10) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/home/jovyan/ntd-snapshot/.venv/lib/python3.11/site-packages/google/api_core/_python_version_support.py:255: FutureWarning: You are using a Python version (3.11.10) which Google will stop supporting in new releases of google.cloud._storage_v2 once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.cloud._storage_v2 past that date.
  warnings.warn(message, FutureWarning)
/home/jovyan/ntd-snapshot/.venv/lib/python3.11/site-packages/google/ap

In [3]:
df_capex, df_opex, df_merged, df_mode_efficiency, df_agency_mode_efficiency  = process_ntd.load_processed_data_for_mode_report()

## Analytical Composition

Analytical Sample Disclaimer: The mode-level analysis includes all available agency observations in the NTD data for each year. Results are not separately segmented by reporter type, reporting module, or agency status in the remaining charts. The composition of the available agency observations by these characteristics is shown in the following composition chart and should be considered when interpreting year-to-year differences in spending, performance, and cost-effectiveness.

In [4]:
sample_chart = mode_report_chart.composition_by_year(
    df_opex,
    {
        "reporter_type": "Reporter Type",
        "reporting_module": "Reporting Module",
        "agency_status": "Agency Status",
        "type_of_service_full_name": "Type of Service"
    },
    chart_title="Analytic Sample Composition by Year"
)

sample_chart

alt.VConcatChart(...)

## 1. Trend & composition

### 1.1 Ridership, OPEX, CAPEX, VRH and VRM Trends

In [5]:
ridership_trend = mode_report_chart.mode_trend_chart(df_opex, "unlinked_passenger_trips", title="Unlinked Passenger Trips", width=900)
opex_trend = mode_report_chart.mode_trend_chart(df_opex, "operating_expenses_total_real", title="Operating Expenditures (Total)")
capex_trend = mode_report_chart.mode_trend_chart(df_capex, "total_capital_expenditures_real", title="Capital Expenditures (Total)")
vrh_trend = mode_report_chart.mode_trend_chart(df_opex, "vehicle_revenue_hours", title="Vehicle Revenue Hours")
vrm_trend = mode_report_chart.mode_trend_chart(df_opex, "vehicle_revenue_miles", title="Vehicle Revenue Miles")

mode_report_grid = (
    alt.vconcat(
        ridership_trend,
        alt.hconcat(opex_trend, capex_trend),
        alt.hconcat(vrh_trend, vrm_trend)
    )
    .resolve_scale(color="shared")
    .configure_view(strokeWidth=0, fill="white")
)

mode_report_grid

/home/jovyan/ntd-snapshot/.venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3699: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  exec(code_obj, self.user_global_ns, self.user_ns)
/home/jovyan/ntd-snapshot/.venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3699: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  exec(code_obj, self.user_global_ns, self.user_ns)
/home/jovyan/ntd-snapshot/.venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3699: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want indepe

alt.VConcatChart(...)

### 1.2 Capital and Operating Expenditures by Mode Group Over Time

In [6]:
capex_opex_share = mode_report_chart.percent_stacked_bar(
    df_merged,
    "mode_group",
    {
        "CAPEX": "total_capital_expenditures_real",
        "OPEX": "operating_expenses_total_real",
    },
    year_range=(2015, 2024),
    chart_title="CAPEX and OPEX by Mode Group",
    chart_subtitle="Mode Groups: Bus, Rail, and Other",
    y_title="Share of Total Expenditure (%)",
)

capex_opex_share

alt.Chart(...)

### 1.3 Expenses Breakdown by Mode for Latest NTD Year

In [7]:
opex_chart = mode_report_chart.breakdown_chart(
    df_opex,
    group_column = "mode_group", 
    year_column = "year",
    component_columns=[
        "operating_expenses_vehicle_operations_real",
        "operating_expenses_vehicle_maintenance_real",
        "operating_expenses_nonvehicle_maintenance_real",
        "operating_expenses_general_administration_real",
    ],
    title = "Operating Expense Composition by Mode",
)

capex_chart = mode_report_chart.breakdown_chart(
    df_capex,
    group_column = "mode_group", 
    year_column = "year",
    component_columns=[
        "rolling_stock_expenditures_real",
        "facilities_expenditures_real",
        "other_expenditures_real",
    ],
    title = "Capital Expenditure Composition by Mode",
)

charts = [opex_chart, capex_chart]

breakdown_by_mode = (
    alt.hconcat(*charts)
    .configure_axis(**mode_report_chart.ALT_AXIS_CONFIG)
    .configure(font = mode_report_chart.CHART_FONT)
)

breakdown_by_mode

alt.HConcatChart(...)

## 2. Efficiency/ Performance

### 2.1 Mode Performance (HHI, Spending Per Capita and Volatility)

In [8]:
scorecard = df_opex[df_opex["year"] == 2024].drop_duplicates("mode_group")

opex_scorecard = mode_report_chart.plot_scorecard(
    scorecard,
    metrics=["hhi", "per_capita", "volatility"],
    group_column="mode_group",
    titles=[
        "Expense Concentration",
        "Average Agency Spending per Capita",
        "Spending Volatility (Compared to Previous Year)",
    ],
    x_labels=[
        "Operating Expenditure Composition HHI",
        "Average Agency Spending per Capita (USD)",
        "Absolute YoY change in total mode-group spending (%)",
    ],
    formats=[".3f", "$,.0f", ".1%"],
    main_title="OPEX Mode Performance 2024",
).configure_axis(**mode_report_chart.ALT_AXIS_CONFIG)

opex_scorecard

alt.HConcatChart(...)

In [9]:
scorecard = df_capex[df_capex["year"] == 2024].drop_duplicates("mode_group")

capex_scorecard = mode_report_chart.plot_scorecard(
    scorecard,
    metrics=["hhi", "per_capita", "volatility"],
    group_column="mode_group",
    titles=[
        "Expense Concentration",
        "Average Agency Spending per Capita",
        "Spending Volatility (Compared to Previous Year)",
    ],
    x_labels=[
        "Capital Expenditure Composition HHI",
        "Average Agency Spending per Capita (USD)",
        "Absolute YoY change in total mode-group spending (%)",
    ],
    formats=[".3f", "$,.0f", ".1%"],
    main_title="CAPEX Mode Performance 2024",
).configure_axis(**mode_report_chart.ALT_AXIS_CONFIG)

capex_scorecard

alt.HConcatChart(...)

### 2.2 Cost-Efficiency by Mode

In [10]:
cost_efficiency_matrix = mode_report_chart.efficiency_matrix(
    df_mode_efficiency,
    metrics = ["opex_per_vrh", "opex_per_vrm", "opex_per_upt"],
    group_column = "mode_full_name",
    year_column = "year",
    titles = ["Opex per VRH", "Opex per VRM", "Opex per Trip"],
    normalize = "zscore",
    reverse_scale = [True, True, True],
    chart_title = "Cost-Efficiency by Mode",
    width = 600,
    height = 400,
)

cost_efficiency_matrix

/home/jovyan/ntd-snapshot/.venv/lib/python3.11/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/home/jovyan/ntd-snapshot/.venv/lib/python3.11/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/home/jovyan/ntd-snapshot/.venv/lib/python3.11/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


alt.Chart(...)

### 2.3 Cost per VRH: Variation Across Agencies Within Each Mode

In [11]:
spread_vrh = mode_report_chart.simple_box_plot(
    df_agency_mode_efficiency,
    value_column = "opex_per_vrh",
    group_column = "mode_full_name",
    year_column = "year",
    value_format = "$,.0f",
    x_title = "Operating Expense per Vehicle Revenue Hour",
    chart_title = "Operating Cost per Revenue Hour: Agency Variation by Mode",
)
spread_vrh

alt.Chart(...)

## 3. Cost-effectiveness

### 3.1 Farebox Recovery Ratio by Mode

In [12]:
farebox_recovery_ridge = mode_report_chart.ridge_plot(
    df_agency_mode_efficiency, "farebox_recovery_ratio", "mode_full_name",
    year_column = "year",
    step = 30, overlap = 1.5,
    chart_title = "Farebox Recovery Ratio Distribution by Transit Mode",
)

farebox_recovery_ridge

alt.FacetChart(...)

### 3.2 Cost Effectiveness Matrix

In [13]:
cost_effectiveness_matrix = mode_report_chart.efficiency_matrix(
    df_mode_efficiency,
    metrics=["farebox_recovery", "fare_revenue_per_trip", "net_subsidy_per_trip"],   
    group_column="mode_full_name",
    year_column = "year",
    titles=["Farebox Recovery", "Fare Revenue per Trip", "Net Subsidy per Trip"],
    normalize="zscore",
    reverse_scale=[False, False, True],
    chart_title="Cost-Effectiveness by Mode",
    width=600,
    height=400
)

cost_effectiveness_matrix

/home/jovyan/ntd-snapshot/.venv/lib/python3.11/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/home/jovyan/ntd-snapshot/.venv/lib/python3.11/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


alt.Chart(...)

### 3.3 Operating Costs vs. Fare Revenue, Indexed to 2019

In [14]:
scissors_chart = mode_report_chart.indexed_scissors(
    df_opex, "mode_group", "year",
    ["operating_expenses_total_real", "fare_revenue_real"], 2019,
    {"operating_expenses_total_real": "Operating Expenses",
     "fare_revenue_real": "Fare Revenue"},
    chart_title = "Real Operating Costs vs. Real Fare Revenue",
    chart_subtitle = "Both indexed to 2019 = 100; widening separation indicates divergent growth",
)

scissors_chart

alt.FacetChart(...)

## 4. Service Effectiveness

### 4.1 Operating Efficiency: Cost vs. Service Utilization and Service Intensity

In [15]:
# VRM
opex_vrm = mode_report_chart.grouped_scatter(
    df_agency_mode_efficiency,
    "log_opex_vrm", "log_trips_vrm",
    group_column="mode_group",
    size_column="unlinked_passenger_trips",
    label_column="source_agency",
    tooltip_columns=["source_agency", "mode_group", "log_opex_vrm", "log_trips_vrm"],
    x_title="Log Operating Expense per Vehicle Revenue Mile",
    y_title="Log Passenger Trips per Vehicle Revenue Mile",
    chart_title="Operating Cost vs. Service Utilization — VRM-based",
)

# VRH
opex_vrh = mode_report_chart.grouped_scatter(
    df_agency_mode_efficiency,
    "log_opex_vrh", "log_trips_vrh",
    group_column="mode_group",
    size_column="unlinked_passenger_trips",
    label_column="source_agency",
    tooltip_columns=["source_agency", "mode_group", "log_opex_vrh", "log_trips_vrh"],
    x_title="Log Operating Expense per Vehicle Revenue Hour",
    y_title="Log Passenger Trips per Vehicle Revenue Hour",
    chart_title="Operating Cost vs. Service Intensity — VRH-based",
)

service_utilization_chart = (
    alt.hconcat(opex_vrm, opex_vrh, spacing=30)
    .configure_view(strokeWidth=0)
    .configure_axis(**mode_report_chart.ALT_AXIS_CONFIG)
    .configure_legend(**mode_report_chart.ALT_LEGEND_CONFIG)
    .configure_title(font=mode_report_chart.CHART_FONT)
)

service_utilization_chart

alt.HConcatChart(...)